In [1]:
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

In [ ]:
from google.colab import files
files.upload()

In [3]:
!ls -lha kaggle.json
!pip install -q kaggle # installing the kaggle package
!mkdir -p ~/.kaggle # creating .kaggle folder where the key should be placed
!cp kaggle.json ~/.kaggle/ # move the key to the folder
!pwd # checking the present working directory

-rw-r--r-- 1 root root 73 May 12 05:43 kaggle.json
/content


In [4]:
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d sujaymann/handwritten-english-characters-and-digits -p /dataset

Dataset URL: https://www.kaggle.com/datasets/sujaymann/handwritten-english-characters-and-digits
License(s): ODC Attribution License (ODC-By)
100% 205M/205M [00:11<00:00, 18.5MB/s]



In [ ]:
!unzip /dataset/handwritten-english-characters-and-digits.zip -d /dataset/handwritten-english-characters-and-digits/

In [8]:
import os

x = len(os.listdir('/dataset/handwritten-english-characters-and-digits/augmented_images/augmented_images1/A_caps'))
y = len(os.listdir('/dataset/handwritten-english-characters-and-digits/handwritten-english-characters-and-digits/combined_folder/train/A_caps'))
print(x, y, x + y)

220 44 264


In [9]:
train_path = '/dataset/handwritten-english-characters-and-digits/handwritten-english-characters-and-digits/combined_folder/train'
augment_path = '/dataset/handwritten-english-characters-and-digits/augmented_images/augmented_images1'

In [10]:
complete_str = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

In [11]:
def get_path(alpha):
    if ord(alpha) < 97 and ord(alpha) >= 65:
        return alpha.upper() + "_caps"
    else:
        return alpha

In [12]:
def prep_img(img):
    resize = img.resize((64, 64)).convert('L')
    img_np = np.array(resize).astype(np.float32)
    img_np /= 255.0

    img_np = np.expand_dims(img_np, axis=0)
    return torch.tensor(img_np)

In [13]:
def get_all_image_paths_for_class(base_path, class_folder_name):
    class_path = os.path.join(base_path, class_folder_name)
    if not os.path.exists(class_path):
        return []

    image_files = []
    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_files.append(os.path.join(class_path, filename))
    return image_files

In [14]:

all_data = []
for class_idx, char in enumerate(complete_str):
    folder_name = get_path(char)

    train_images = get_all_image_paths_for_class(train_path, folder_name)
    for img_path in train_images:
        all_data.append((img_path, class_idx))

    augment_images = get_all_image_paths_for_class(augment_path, folder_name)
    for img_path in augment_images:
        all_data.append((img_path, class_idx))

print(f"Total images collected: {len(all_data)}")

random.shuffle(all_data)

num_folds = 4

folds = [[] for _ in range(num_folds)]
for i, (img_path, label) in enumerate(all_data):
    fold_id = i % num_folds
    folds[fold_id].append((img_path, label))

print(f"Data distributed into {num_folds} folds.")
for i, fold in enumerate(folds):
    print(f"Fold {i}: {len(fold)} images")

Total images collected: 16368
Data distributed into 4 folds.
Fold 0: 4092 images
Fold 1: 4092 images
Fold 2: 4092 images
Fold 3: 4092 images


In [15]:
merge_output_dir = '/dataset/merge'
os.makedirs(merge_output_dir, exist_ok=True)

for i, fold_data in enumerate(folds):
    images_tensor_list = []
    labels_tensor_list = []

    print(f"Processing Fold {i}...")
    for img_path, label in fold_data:
        try:
            img = Image.open(img_path)
            processed_img = prep_img(img)
            images_tensor_list.append(processed_img)
            labels_tensor_list.append(label)
        except Exception as e:
            print(f"Error processing image {img_path}: {e}")

    if images_tensor_list:
        images_tensor = torch.stack(images_tensor_list)
        labels_tensor = torch.tensor(labels_tensor_list)

        torch.save({
            "images": images_tensor,
            "labels": labels_tensor
        }, os.path.join(merge_output_dir, f"fold_{i}.pt"))
        print(f"Saved {len(images_tensor_list)} items to {os.path.join(merge_output_dir, f'fold_{i}.pt')}")
    else:
        print(f"No valid images processed for Fold {i}.")

print("All folds processed and saved.")

Processing Fold 0...
Saved 4092 items to /dataset/merge/fold_0.pt
Processing Fold 1...
Saved 4092 items to /dataset/merge/fold_1.pt
Processing Fold 2...
Saved 4092 items to /dataset/merge/fold_2.pt
Processing Fold 3...
Saved 4092 items to /dataset/merge/fold_3.pt
All folds processed and saved.


In [16]:
class MyCNN(nn.Module):

    def __init__(self, num_classes=62):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(
            32,
            64,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(
            64,
            128,
            kernel_size=3,
            padding=1
        )

        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool(x)

        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


In [17]:
def calc_accuracy(outputs, labels):
    _, preds = torch.max(outputs, 1)

    correct = (preds == labels).sum().item()
    accuracy = correct / labels.size(0)

    return accuracy

In [18]:
def train_model(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    running_acc = 0.0

    for batch in loader:

        images = batch["images"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        acc = calc_accuracy(outputs, labels)

        running_loss += loss.item()
        running_acc += acc

    epoch_loss = running_loss / len(loader)
    epoch_acc = running_acc / len(loader)

    return epoch_loss, epoch_acc

In [19]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    running_acc = 0.0

    with torch.no_grad():

        for batch in loader:

            images = batch["images"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            acc = calc_accuracy(outputs, labels)

            running_loss += loss.item()
            running_acc += acc

    epoch_loss = running_loss / len(loader)
    epoch_acc = running_acc / len(loader)

    return epoch_loss, epoch_acc

In [20]:
class CharacterDataset(Dataset):

    def __init__(self, images, labels):

        self.images = images.float()
        self.labels = labels.long()

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        image = self.images[idx]
        label = self.labels[idx]

        return {
            "images": image,
            "labels": label
        }

In [22]:
folds = []

for i in range(4):

    data = torch.load(f"/dataset/merge/fold_{i}.pt")

    folds.append(data)

In [28]:
def create_fold_datasets(folds, fold_idx):

    train_images = []
    train_labels = []

    val_images = folds[fold_idx]["images"]
    val_labels = folds[fold_idx]["labels"]

    for i in range(4):

        if i != fold_idx:

            train_images.append(folds[i]["images"])
            train_labels.append(folds[i]["labels"])

    train_images = torch.cat(train_images, dim=0)
    train_labels = torch.cat(train_labels, dim=0)

    train_dataset = CharacterDataset(
        train_images,
        train_labels
    )

    val_dataset = CharacterDataset(
        val_images,
        val_labels
    )

    return train_dataset, val_dataset

In [24]:
def create_dataloaders(
    train_dataset,
    val_dataset,
    batch_size=32
):

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2
    )

    return train_loader, val_loader

In [25]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MyCNN(num_classes=62).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [26]:
print(torch.cuda.is_available())

True


In [29]:
train_dataset, val_dataset = create_fold_datasets(
    folds,
    fold_idx=0
)

train_loader, val_loader = create_dataloaders(
    train_dataset,
    val_dataset,
    batch_size=32
)

In [30]:
best_val_loss = float("inf")

In [ ]:
EPOCHS = 40
for fold_idx in range(4):

    print(f"\n========== FOLD {fold_idx} ==========")

    train_dataset, val_dataset = create_fold_datasets(
        folds,
        fold_idx
    )

    train_loader, val_loader = create_dataloaders(
        train_dataset,
        val_dataset
    )

    model = MyCNN(num_classes=62).to(device)

    print("Loaded Saved model")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):

        train_loss, train_acc = train_model(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        val_loss, val_acc = validate(
            model,
            val_loader,
            criterion,
            device
        )

        print(f"Epoch {epoch+1}/{EPOCHS}")

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Acc : {train_acc:.4f}")

        print(f"Val Loss  : {val_loss:.4f}")
        print(f"Val Acc   : {val_acc:.4f}")




        # SAVE BEST MODEL
        if val_loss < best_val_loss:

            best_val_loss = val_loss

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_acc": val_acc
            }, "best_model.pth")

            print("Best model saved!")

In [32]:
test_path='/dataset/handwritten-english-characters-and-digits/handwritten-english-characters-and-digits/combined_folder/test'

In [33]:
model = MyCNN(num_classes=62).to(device)
checkpoint = torch.load("best_model.pth")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best model loaded successfully!")

Best model loaded successfully!


In [34]:

test_data_list = []
for class_idx, char in enumerate(complete_str):
    folder_name = get_path(char)
    test_images_paths = get_all_image_paths_for_class(test_path, folder_name)
    for img_path in test_images_paths:
        test_data_list.append((img_path, class_idx))

print(f"Total test images collected: {len(test_data_list)}")


test_images_tensor_list = []
test_labels_tensor_list = []

print("Processing test images...")
for img_path, label in test_data_list:
    try:
        img = Image.open(img_path)
        processed_img = prep_img(img)
        test_images_tensor_list.append(processed_img)
        test_labels_tensor_list.append(label)
    except Exception as e:
        print(f"Error processing test image {img_path}: {e}")

if test_images_tensor_list:
    test_images_tensor = torch.stack(test_images_tensor_list)
    test_labels_tensor = torch.tensor(test_labels_tensor_list)
    print(f"Successfully processed {len(test_images_tensor_list)} test images.")
else:
    print("No valid test images processed.")


test_dataset = CharacterDataset(
    test_images_tensor,
    test_labels_tensor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Test dataset and dataloader created.")

Total test images collected: 682
Processing test images...
Successfully processed 682 test images.
Test dataset and dataloader created.


In [35]:
def test_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_acc = 0.0

    with torch.no_grad():
        for batch in loader:
            images = batch["images"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            acc = calc_accuracy(outputs, labels)

            running_loss += loss.item()
            running_acc += acc

    avg_loss = running_loss / len(loader)
    avg_acc = running_acc / len(loader)

    return avg_loss, avg_acc

test_loss, test_acc = test_model(model, test_loader, criterion, device)

print(f"\n========== TEST RESULTS ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")


========== TEST RESULTS ==========
Test Loss: 0.4881
Test Acc : 0.8545


In [36]:
torch.save(model.state_dict(), "production_model.pth")
print("Model saved to production_model.pth")

Model saved to production_model.pth


You can then load this model in a production environment as follows:

```python
# In your production environment

# 1. Instantiate your model architecture (MyCNN should be defined or imported)
# model_for_production = MyCNN(num_classes=62)

# 2. Load the saved state dictionary
# model_for_production.load_state_dict(torch.load("production_model.pth"))

# 3. Set the model to evaluation mode
# model_for_production.eval()

# Now, model_for_production is ready to make predictions
```